In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.parse as urlparse
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from bs4 import BeautifulSoup
import requests
pd.options.display.float_format = '{:.5f}'.format

In [2]:
df = pd.read_csv('https://github.com/frohsch/raw/한국전력거래소_시간별 전력수요량_20211231.csv', thousands = ',', encoding= 'cp949')
df


InvalidURL: URL can't contain control characters. '/frohsch/raw/한국전력거래소_시간별 전력수요량_20211231.csv' (found at least ' ')

In [ ]:
df.rename(columns = {'거래일자' : 'date'}, inplace = True)

In [ ]:
df["date"] = pd.to_datetime(df["date"])

In [ ]:
df['total'] = df[df.columns[1:25]].sum(axis=1)
df['am'] = df[df.columns[1:12]].sum(axis=1)
df['pm'] = df[df.columns[13:25]].sum(axis=1)   

In [ ]:
df["year"] = df["date"].dt.year # 월
df["month"] = df["date"].dt.month # 월
df["day"] = df["date"].dt.day # 일
df["weekday"] = df["date"].dt.weekday # 0: 월요일
df["weekend"] = df["weekday"].isin([5, 6]).astype(int) # 5: 토요일
df['dayofyear'] = df["date"].dt.dayofyear #연 기준 며칠째인지

df.head()

In [ ]:
# 계절 컬럼 추가 
def season(month): 
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    elif month in [9, 10, 11]:
        return "fall"

df["season"] = df.month.apply(season)

In [ ]:
# 토, 일을 제외한 국경일 및 공휴일 추가
url = "http://apis.data.go.kr/B090041/openapi/service/SpcdeInfoService"
operation = "getRestDeInfo"
mykey = 'ac%2FSOW4KzOFKdu0z01iEVWGZd4TBl1MyiR04%2FfYmADthCjJBEyL73pewbapUk94Gm1%2FqtzMCban3C%2BpqU8c7ew%3D%3D'

date = []
datename = []
for year in range(2007, 2021):
    year = str(year)
    
    for month in range(1, 13):
        if month < 10:
            month = "0" + str(month)
        else:
            month = str(month)
            
        params = {'solYear' : year, 'solMonth' : month}
        rq_query = url +'/' + operation + '?' + urlparse.urlencode(params) + "&serviceKey=" + mykey    
        response = requests.get(rq_query) 
        dom = BeautifulSoup(response.content, "html.parser")
        
        items = dom.find_all("item")
        for item in items:
            date.append(item.locdate.string)
            datename.append(item.datename.string)

holiday_df= pd.DataFrame({"date": date, "datename": datename})

In [ ]:
holiday_df["datename"].unique()

In [ ]:
holiday_df.loc[holiday_df["datename"].isin(["신정", "1월1일"]),"datename"] = "신정"
holiday_df.loc[holiday_df["datename"].isin(["부처님오신날", "석가탄신일"]),"datename"] = "석가탄신일"
holiday_df.loc[holiday_df["datename"].isin(["국회의원선거일", "대통령선거일", "전국동시지방선거"]),"datename"] = "선거일"
holiday_df.loc[holiday_df["datename"].isin(["대체공휴일", "대체휴무일", "임시공휴일"]),"datename"] = "대체공휴일"

In [ ]:
holiday_df

In [ ]:
holiday_df.to_pickle("./holiday_df.pkl")

In [ ]:
holiday_df = pd.read_pickle("./holiday_df.pkl")

In [ ]:
holiday_df.info()

In [ ]:
holiday_df['date']= holiday_df['date'].astype('str')

In [ ]:
holiday_df2 = pd.DataFrame({'datetime' : [str(i) for i in holiday_df['date']]})

In [ ]:
holiday_df['date']= holiday_df2['datetime']

In [ ]:
holiday_df["date"] = pd.to_datetime(holiday_df["date"])

In [ ]:
holiday_df

In [ ]:
# 기존 데이터와 공휴일 데이터 합치기 
df["date"] = df.date.astype(str)
holiday_df["date"] = holiday_df.date.astype(str)
df = pd.merge(df, holiday_df, how = "left")
df.loc[df["datename"].notnull(), "weekend"] = 1

In [ ]:
df

In [ ]:
df.datename.fillna('평일', inplace=True)

In [ ]:
df

In [ ]:
df.describe()

In [ ]:
df.isnull().sum().to_frame('nan_count')